# 8장. 작은 데이터 분석 프로젝트 완성하기

이 노트북은 데이터 점검, 전처리, 병합 검증, 완료 주문 기준 집계, 시각화, 보고서 작성을 하나의 재현 가능한 프로젝트로 연결합니다.


## 학습 목표

- 원본 데이터를 보존하며 전처리합니다.
- 주요 키 중복과 외래키 미매칭을 확인합니다.
- `validate`와 `indicator`를 사용한 병합 검증 원리를 확인합니다.
- 취소·환불 주문을 제외하고 완료 주문 기준 매출을 계산합니다.
- 고객 이름을 노출하지 않는 익명화 결과를 만듭니다.
- 결과 CSV, 그래프, Markdown 보고서를 재현 가능하게 저장합니다.


## 1. 프로젝트 루트와 출력 폴더 설정

VS Code에서 현재 작업 폴더가 프로젝트 루트 또는 `notebooks`일 수 있으므로 상위 폴더를 탐색합니다.


In [ ]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
FIGURE_DIR = REPORT_DIR / 'figures'

for path in [PROCESSED_DIR, REPORT_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Python 실행 파일:', sys.executable)
print('프로젝트 루트:', PROJECT_ROOT)
print('원본 데이터 폴더:', RAW_DIR)
print('보고서 폴더:', REPORT_DIR)


## 2. 프로젝트 함수 불러오기

중간 프로젝트의 공통 로직은 `src/` 모듈에 정리되어 있습니다. Notebook과 일괄 실행 스크립트가 같은 함수를 사용하므로 계산 기준이 어긋나는 문제를 줄일 수 있습니다.


In [ ]:
from src.data_loader import load_sales_data
from src.preprocessing import (
    compare_shapes,
    preprocess_sales_data,
    save_processed_data,
    validate_relationships,
)
from src.midterm_project import (
    build_analysis_tables,
    build_interpretation_notes,
    build_key_duplicate_checks,
    build_midterm_report,
    create_project_figures,
    run_midterm_project,
    save_project_tables,
    summarize_datasets,
)


## 3. 원본 데이터 확인

파일이 없다면 프로젝트 루트에서 `python scripts/generate_sample_data.py`를 먼저 실행합니다.


In [ ]:
raw_data = load_sales_data(RAW_DIR)

dataset_summary = summarize_datasets(raw_data)
display(dataset_summary)

for name, df in raw_data.items():
    print(f'\n===== {name} =====')
    print('shape:', df.shape)
    print('columns:', df.columns.tolist())
    print('missing values:', int(df.isna().sum().sum()))
    print('duplicated rows:', int(df.duplicated().sum()))


## 4. 원본을 보존하며 전처리

`preprocess_sales_data()`는 원본 DataFrame을 직접 수정하지 않고 전처리된 복사본을 반환합니다.


In [ ]:
processed_data = preprocess_sales_data(raw_data)
processed_paths = save_processed_data(processed_data, PROCESSED_DIR)

preprocessing_comparison = compare_shapes(raw_data, processed_data)
display(preprocessing_comparison)

for path in processed_paths:
    print(path, 'OK' if path.exists() else 'MISSING')


## 5. 키 중복과 관계 점검

전체 행 중복뿐 아니라 고객·상품·주문·주문 상세의 식별 키 중복을 확인합니다. 외래키가 기준 테이블에 존재하지 않는 행도 확인합니다.


In [ ]:
key_duplicate_checks = build_key_duplicate_checks(processed_data)
relationship_checks = validate_relationships(processed_data)

display(key_duplicate_checks)
display(relationship_checks)


## 6. 안전한 병합과 완료 주문 기준 집계

`build_analysis_tables()`는 다음 기준을 적용합니다.

- 주문 상세와 주문: `many_to_one`
- 완료 주문만 매출 집계에 포함
- 주문 상세와 상품: `many_to_one`
- 고객 집계와 고객 속성: `one_to_one`
- 병합 전후 행 수와 미매칭 건수 기록


In [ ]:
analysis_tables = build_analysis_tables(processed_data)

display(analysis_tables['merge_checks'])
display(analysis_tables['amount_scope_summary'])


## 7. 주요 결과 확인

아래 매출표는 모두 완료 주문 기준입니다. 주문 상태별 요약표는 전체 주문을 대상으로 합니다.


In [ ]:
category_sales = analysis_tables['category_sales']
monthly_sales = analysis_tables['monthly_sales']
customer_sales = analysis_tables['customer_sales']
order_status_summary = analysis_tables['order_status_summary']

display(category_sales)
display(monthly_sales)
display(
    customer_sales[
        ['customer_label', 'city', 'order_count', 'total_sales', 'avg_order_value']
    ].head(10)
)
display(order_status_summary)


## 8. 해석 메모와 결과 저장

관찰, 주의점, 다음 질문을 분리합니다. 고객 결과에는 실제 이름을 포함하지 않습니다.


In [ ]:
interpretation_notes = build_interpretation_notes()
display(interpretation_notes)

saved_tables = save_project_tables(
    dataset_summary,
    preprocessing_comparison,
    key_duplicate_checks,
    relationship_checks,
    analysis_tables,
    interpretation_notes,
    REPORT_DIR,
)

for path in saved_tables:
    print(path.name, path.exists(), path.stat().st_size)


## 9. 그래프 생성

그래프 제목에도 완료 주문 기준임을 표시합니다.


In [ ]:
saved_figures = create_project_figures(
    analysis_tables,
    FIGURE_DIR,
    show=True,
)

for path in saved_figures:
    print(path.name, path.exists(), path.stat().st_size)


## 10. Markdown 보고서 생성

보고서는 완료 주문 기준, 병합 검증, 익명화 원칙을 명시합니다.


In [ ]:
report_text = build_midterm_report(
    dataset_summary,
    preprocessing_comparison,
    key_duplicate_checks,
    relationship_checks,
    analysis_tables,
    interpretation_notes,
)

report_path = REPORT_DIR / 'ch08_midterm_report.md'
report_path.write_text(report_text, encoding='utf-8')

print('보고서 저장:', report_path)
print(report_text[:1500])


## 11. 전체 파이프라인 재실행

단계별 결과를 확인한 뒤에는 같은 함수를 사용하는 전체 파이프라인으로 재현성을 점검합니다.


In [ ]:
project_result = run_midterm_project(
    raw_dir=RAW_DIR,
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
    figure_dir=FIGURE_DIR,
    show_figures=False,
)

project_result['report_path']


## 12. 최종 산출물 검증

필수 파일이 존재하고 비어 있지 않은지 확인합니다.


In [ ]:
expected_outputs = [
    REPORT_DIR / 'ch08_midterm_report.md',
    REPORT_DIR / 'ch08_dataset_summary.csv',
    REPORT_DIR / 'ch08_preprocessing_comparison.csv',
    REPORT_DIR / 'ch08_key_duplicate_checks.csv',
    REPORT_DIR / 'ch08_relationship_checks.csv',
    REPORT_DIR / 'ch08_merge_checks.csv',
    REPORT_DIR / 'ch08_amount_scope_summary.csv',
    REPORT_DIR / 'ch08_category_sales.csv',
    REPORT_DIR / 'ch08_monthly_sales.csv',
    REPORT_DIR / 'ch08_customer_sales.csv',
    REPORT_DIR / 'ch08_order_status_summary.csv',
    FIGURE_DIR / 'ch08_category_sales.png',
    FIGURE_DIR / 'ch08_monthly_sales.png',
    FIGURE_DIR / 'ch08_top_customers.png',
]

all_outputs_valid = True
for path in expected_outputs:
    valid = path.exists() and path.stat().st_size > 0
    all_outputs_valid &= valid
    print(path.name, 'OK' if valid else 'MISSING OR EMPTY')

print('전체 산출물 검증:', all_outputs_valid)


## 13. 개인정보와 결과 일치 검증

고객 이름이 결과표에 포함되지 않았는지 확인하고, 카테고리 매출 합계와 완료 주문 매출 합계가 일치하는지 확인합니다.


In [ ]:
assert 'name' not in customer_sales.columns

completed_total = analysis_tables['completed_order_sales']['line_total'].sum()
category_total = category_sales['total_sales'].sum()
monthly_total = monthly_sales['total_sales'].sum()
customer_total = customer_sales['total_sales'].sum()

print('완료 주문 매출:', completed_total)
print('카테고리 합계:', category_total)
print('월별 합계:', monthly_total)
print('고객별 합계:', customer_total)

assert completed_total == category_total == monthly_total == customer_total
print('결과 합계 일치 검증 완료')


## 14. LLM 검토 프롬프트 예시

```text
온라인 쇼핑몰 중간 프로젝트 결과를 검토해 주세요.

분석 기준:
- 매출은 order_status가 completed인 주문만 포함
- 고객 이름은 제외하고 익명화 라벨 사용
- 병합은 validate와 indicator로 검증

검토 항목:
1. 분석 질문과 지표가 연결되어 있는가?
2. 취소·환불 주문이 매출에 포함되지 않았는가?
3. 병합 후 행 증가 또는 미매칭을 확인했는가?
4. 관찰과 원인 가설을 구분했는가?
5. 개인정보가 불필요하게 노출되지 않았는가?
6. Notebook, CSV, 그래프, 보고서의 수치가 일치하는가?

데이터에 없는 원인은 단정하지 말고,
필수 수정과 권장 개선을 구분해 주세요.
```


## 15. 실습 과제

1. 카테고리별 평균 판매 단가를 추가합니다.
2. 월별 취소율과 환불률을 계산합니다.
3. 고객별 최근 구매일과 구매 빈도를 추가합니다.
4. 전체 주문 상세 금액과 완료 주문 매출의 차이를 해석합니다.
5. 결과 합계가 모든 집계표에서 일치하는지 검증 코드를 추가합니다.
6. LLM 활용 내역과 실제 반영 여부를 표로 작성합니다.


In [ ]:
# 실습 과제 코드를 아래에 작성하세요.


## 정리

이번 장에서는 원본 데이터 확인, 전처리, 키 관계 점검, 병합 검증, 완료 주문 기준 매출 계산, 익명화, 시각화, 보고서 생성, 산출물 검증을 하나의 프로젝트로 연결했습니다. 다음 장부터는 이 분석 기반 위에서 머신러닝으로 확장합니다.
